# SER - Experiment 04: augment-before-split leakage

**Question:** does augmenting *before* splitting account for the rest of the
gap to the base paper's 94.91%?

Experiment 03 showed duplicate mirrors are worth **+14.71 points**
(0.5795 -> 0.7266) - large, but 22 points short of 0.9491.

This notebook tests the second and more common mechanism. The canonical broken
pattern in published SER pipelines is:

```python
for path, emotion in data.iterrows():
    features = get_features(path)     # original + noise + stretch + pitch
    for f in features:
        X.append(f); y.append(emotion)

x_train, x_test = train_test_split(X, y, ...)   # <-- split AFTER augmenting
```

Every utterance becomes several near-identical rows. A random split then puts
siblings of the same recording on both sides of the boundary.

Here the corpus is the **clean, de-duplicated 12,162 samples** - so this
isolates the augment-before-split effect on its own, with no duplicate-mirror
contamination mixed in.

| Run | Order | Corpus | Accuracy |
|---|---|---|---|
| `base` | split -> augment (correct) | 12,162 | 0.5795 |
| `base_leaky` | duplicated mirror | 16,402 | 0.7266 |
| `base_augleak` (this) | **augment -> split** | 12,162 | ? |

**Attach:** the four corpora only.
**Accelerator: GPU.** Roughly 25 min extraction plus 20 min training.

In [ ]:
import glob
import json
import os
import shutil
import sys
import time

import numpy as np
import tensorflow as tf
import keras

print("TF", tf.__version__, "| Keras", keras.__version__)
gpus = tf.config.list_physical_devices("GPU")
print("GPUs:", gpus)
assert gpus, "Set Settings -> Accelerator -> GPU before running."

In [ ]:
REPO = "https://github.com/Eldorado5002/ser.git"

if not os.path.exists("/kaggle/working/ser"):
    !git clone -q {REPO} /kaggle/working/ser

sys.path.insert(0, "/kaggle/working/ser")
os.chdir("/kaggle/working/ser")
!git log --oneline -1

In [ ]:
# CLEAN corpus: canonical subdirectories only, exactly as notebook 02 uses.
# This experiment isolates the augment-before-split effect, so there must be
# no duplicate-mirror contamination mixed in.
DATA_ROOT = "/kaggle/working/ser/data"

CANONICAL = {
    "RAVDESS": "audio_speech_actors_01-24",
    "TESS":    "TESS Toronto emotional speech set data",
    "SAVEE":   "ALL",
    "CREMA-D": "AudioWAV",
}


def find_canonical(target):
    hits = []
    for root, dirs, _ in os.walk("/kaggle/input"):
        for d in dirs:
            if d.lower() == target.lower():
                hits.append(os.path.join(root, d))
    return sorted(hits)[0] if hits else None


os.makedirs(DATA_ROOT, exist_ok=True)
for name, target in CANONICAL.items():
    src = find_canonical(target)
    assert src is not None, f"MISSING INPUT for {name}: no '{target}' found"
    dst = os.path.join(DATA_ROOT, name)
    if os.path.islink(dst):
        os.unlink(dst)
    elif os.path.exists(dst):
        shutil.rmtree(dst)
    os.symlink(src, dst)
    print(f"{name:9s} -> {src}")

In [ ]:
import config
from data_loader import build_metadata

config.CACHE_DIR = "/kaggle/working/features_cache_augleak"
config.RUNS_DIR = "/kaggle/working/runs"
os.makedirs(config.CACHE_DIR, exist_ok=True)
os.makedirs(config.RUNS_DIR, exist_ok=True)

meta = build_metadata(strict=True)
assert len(meta) == 12162, f"expected the clean 12162, got {len(meta)}"

# ---------------------------------------------------------------------------
# THE DELIBERATE MISTAKE: augment EVERY utterance FIRST, then split.
# ---------------------------------------------------------------------------
N_COPIES = 2          # each utterance -> 1 original + 2 augmented rows
rng = np.random.default_rng(config.RANDOM_SEED)

items = []
for i, r in enumerate(meta.itertuples()):
    items.append({"path": r.path, "emotion": r.emotion, "augment": False,
                  "emotion_aware": False, "seed": 0, "src": r.path})
    for k in range(N_COPIES):
        items.append({"path": r.path, "emotion": r.emotion, "augment": True,
                      "emotion_aware": False,
                      "seed": int(config.RANDOM_SEED + 1 + i * N_COPIES + k),
                      "src": r.path})

print(f"utterances        : {len(meta)}")
print(f"rows after augment: {len(items)}   ({1 + N_COPIES} per utterance)")

In [ ]:
from sklearn.model_selection import train_test_split

y_all = [it["emotion"] for it in items]

train_items, test_items = train_test_split(
    items, test_size=config.TEST_FRACTION, stratify=y_all,
    random_state=config.RANDOM_SEED)
train_items, val_items = train_test_split(
    train_items, test_size=config.VAL_FRACTION_OF_TRAINVAL,
    stratify=[it["emotion"] for it in train_items],
    random_state=config.RANDOM_SEED)

print(f"[split] train={len(train_items)}  val={len(val_items)}  "
      f"test={len(test_items)}")

# Measure the contamination: how many TEST rows come from a source utterance
# that also appears somewhere in train or val?
seen_src = {it["src"] for it in train_items} | {it["src"] for it in val_items}
leaked = sum(1 for it in test_items if it["src"] in seen_src)

print()
print(f"test rows                        : {len(test_items)}")
print(f"whose source appears in train/val: {leaked}")
print(f"CONTAMINATION                    : "
      f"{100 * leaked / len(test_items):.1f}% of the test set")

In [ ]:
from features import build_feature_matrix
from utils import StreamScalers, set_seed

set_seed(config.RANDOM_SEED)

train_feats = build_feature_matrix(train_items, desc="augleak_train")
val_feats = build_feature_matrix(val_items, desc="augleak_val")
test_feats = build_feature_matrix(test_items, desc="augleak_test")

scalers = StreamScalers().fit(train_feats)
x_train = scalers.transform(train_feats)
x_val = scalers.transform(val_feats)
x_test = scalers.transform(test_feats)

y_train = tf.keras.utils.to_categorical(train_feats["y"], config.NUM_CLASSES)
y_val = tf.keras.utils.to_categorical(val_feats["y"], config.NUM_CLASSES)

print("train:", x_train[0].shape, "val:", x_val[0].shape,
      "test:", x_test[0].shape)

In [ ]:
from model import build_model

run_dir = os.path.join(config.RUNS_DIR, "base_augleak")
os.makedirs(run_dir, exist_ok=True)
ckpt_path = os.path.join(run_dir, "best_model.keras")

# Identical to the `base` configuration: no novelties, plain cross-entropy.
model, _ = build_model(use_afw=False, use_mstc=False)
model.compile(optimizer=tf.keras.optimizers.Adam(config.LEARNING_RATE),
              loss="categorical_crossentropy", metrics=["accuracy"])

callbacks = [
    tf.keras.callbacks.ModelCheckpoint(ckpt_path, monitor="val_loss",
                                       mode="min", save_best_only=True,
                                       verbose=0),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=config.EARLY_STOPPING_PATIENCE,
        restore_best_weights=True, verbose=1),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=config.REDUCE_LR_FACTOR,
        patience=config.REDUCE_LR_PATIENCE, min_lr=config.MIN_LR, verbose=0),
    tf.keras.callbacks.CSVLogger(os.path.join(run_dir, "training_log.csv")),
]

history = model.fit(x_train, y_train, validation_data=(x_val, y_val),
                    epochs=config.EPOCHS, batch_size=config.BATCH_SIZE,
                    callbacks=callbacks, verbose=2)

In [ ]:
from evaluate import evaluate_predictions

# Evaluate the checkpointed best model, matching train.py's behaviour.
if os.path.exists(ckpt_path):
    best = tf.keras.models.load_model(ckpt_path, compile=False)
    model.set_weights(best.get_weights())

y_prob = model.predict(x_test, batch_size=config.BATCH_SIZE, verbose=0)
metrics = evaluate_predictions(test_feats["y"], y_prob, out_dir=run_dir,
                               prefix="test")

In [ ]:
CLEAN_BASE = 0.579531   # notebook 02  - split then augment (correct)
DUP_LEAK = 0.726608     # notebook 03  - duplicated mirrors
PAPER = 0.9491          # Chourasia et al. (2026)

augleak = metrics["accuracy"]

print("=" * 70)
print("  LEAKAGE ACCOUNTING")
print("=" * 70)
print(f"  base         split -> augment,  12,162 rows : {CLEAN_BASE:.4f}")
print(f"  base_leaky   duplicate mirrors, 16,402 rows : {DUP_LEAK:.4f}"
      f"   ({100 * (DUP_LEAK - CLEAN_BASE):+.2f} pts)")
print(f"  base_augleak augment -> split,  {len(items):,} rows : {augleak:.4f}"
      f"   ({100 * (augleak - CLEAN_BASE):+.2f} pts)")
print("-" * 70)
print(f"  base paper reported                         : {PAPER:.4f}")
print("=" * 70)
print()

if augleak >= 0.90:
    print("  CONFIRMED. Augmenting before splitting alone reproduces the")
    print("  literature's accuracy range. The base paper's 94.91% is")
    print("  consistent with this pipeline ordering, not with genuine")
    print("  generalisation.")
elif augleak >= 0.80:
    print("  STRONG. Augment-before-split inflates accuracy substantially and,")
    print("  combined with duplicate mirrors, accounts for most of the gap.")
elif augleak > CLEAN_BASE + 0.05:
    print("  PARTIAL. It inflates results but does not reach the reported")
    print("  range on its own.")
else:
    print("  NOT CONFIRMED. Split ordering is not the driver here.")

json.dump({"clean_base": CLEAN_BASE, "dup_leak": DUP_LEAK,
           "augment_before_split": float(augleak), "paper": PAPER,
           "n_rows": int(len(items)), "copies_per_utterance": 1 + N_COPIES},
          open("/kaggle/working/augleak_experiment.json", "w"), indent=2)

In [ ]:
for name in CANONICAL:
    link = os.path.join(DATA_ROOT, name)
    if os.path.islink(link):
        os.unlink(link)

print("output:", sorted(os.listdir("/kaggle/working")))